In [2]:
import json_tricks
import copy
import numpy as np
import random
import numpy


numpy.random.seed(0)
random.seed(0)


inputs1 = json_tricks.load(open('inputs1.json'))
inputs2 = json_tricks.load(open('inputs2.json'))

answer = {}


In [3]:
numpy.random.seed(0)
random.seed(0)

# Implementing Backpropagation package for Numpy

In this project, we will implement the backpropagation algorithm for numpy package.
Your task will be to implement all the methods of the Node class that will be a wrapper for numpy arrays
supporting backpropagation.

Step-by-step, we will implement the following methods:

1. `__init__` (a constructor)
3. `backward` (a recursive mechanism that triggers backpropagation)
4. `__neg__` (negation operator $- x$)
5. `__add__` (addition operator $x + y$)
6. `__mul__` (product operator $x \cdot y$)
7. `__sub__` (substitution operator $x - y$)
8. `__truediv__` (division operator $x / y$)
9. `exp` (exponentiation $\exp(x)$)
10. `sum` (summation $\sum_{k} x_k$)
11. `matmul` (matrix product $XY$)

After that we will use the implemented methods on:
1. Simple graph from the previous task
2. Two-layer neural network

Let's start!

In [17]:
def broadcast_reduce(grad, target_shape, target_dtype=None):
    """
    Reduce gradient to match target shape, handling broadcasting.
    This is needed when gradients need to be reduced due to broadcasting in forward pass.
    """
    # Start with the gradient as is
    result = grad
    
    # Handle scalar case
    if target_shape == ():
        result = np.array([np.sum(result)])
        if target_dtype:
            result = result.astype(target_dtype)
        return result
    
    # If shapes are already the same, no reduction needed
    if result.shape == target_shape:
        if target_dtype:
            return result.astype(target_dtype)
        return result
    
    # Sum over extra dimensions that were broadcasted
    # First, sum over leading dimensions that don't exist in target
    ndims_added = len(result.shape) - len(target_shape)
    for i in range(ndims_added):
        result = np.sum(result, axis=0)
    
    # Then sum over dimensions that were size 1 in target but expanded in result
    for i, (grad_dim, target_dim) in enumerate(zip(result.shape, target_shape)):
        if target_dim == 1 and grad_dim > 1:
            result = np.sum(result, axis=i, keepdims=True)
    
    # Ensure result is always a numpy array
    if not isinstance(result, np.ndarray):
        result = np.array(result)
    
    # Ensure result has the right shape
    if result.shape != target_shape:
        if result.size == 1 and len(target_shape) == 1 and target_shape[0] == 1:
            result = np.array([result.item()])
    
    # Apply target dtype if specified
    if target_dtype:
        result = result.astype(target_dtype)
    
    return result

`Node([1]) + Node([2]) -> Node`
`(x + y).backward()`

# Task 1

Implement the `__init__` method for the `Node` class:
- create field `data` and assign it to the `data` argument (we will need it to compute the gradient)
- create field `grad` and assign it to the `None` (it will be used to store the gradient of the node)
- create field `inputs` and assign it to the empty list (it will be used to store the nodes on which the current node depends)
- create field `n_dependents` and assign it to the 0 (it will be used to count the number of nodes that depend on the current node)
- create field `n_grads` and assign it to the 0 (it will be used to count the number of gradients that are coming to the current node)

Implement the method `update_grad` for the `Node` class. The point of this method is to take care of the gradients that are coming from the next nodes and accumulate them:
- firstly, create a deep copy of `grad_output` (just in case): `grad_output = copy.deepcopy(grad_output)`
- increase the `n_grads` counter by 1 as one of the gradients of the dependents is registered
- if `grad_output` is `None` and the size of `data` is 1, assign `grad_output` to the numpy array `[1]` (as we can calculate derivatives only of scalar values without any dependencies)
- if `grad_output` is `None`, raise the `BaseException` with the message "Backpropagation is impossible" (as we can't compute the gradient without the gradient from the next node except for the case of a scalar value)
- if `self.grad` is `None`, assign `grad_output` to the `self.grad` (that means that if there was no gradient before, we will just assign the one that we got from the next node)
- otherwise, add `grad_output` to the `self.grad` (that means that if there was a gradient before, we will add the one that we got from the next node to the accumulated value)

Implement the `backward` method for the `Node` class. This method is the heart of the backpropagation algorithm. And it logic is the following:
- it waits and collects gradients from all the dependents of the current node
- then it calculates the list of gradients for the inputs of the current node using the `backward_step` method (this method we will implement for every operation by hand)
- triggers the `backward` method of the inputs with the calculated gradient in the current node as an argument

As a formula, it looks like this:
$\partial_w {L} = \sum_{k=1}^{n} \partial_w {z_k} \partial_{z_k} {L}$

So, how this works in practice?
- call the `update_grad` method with the `grad_output` argument
- if we have collected all the gradients from the dependents, it is time to calculate the gradients for the inputs of the current node 
- call the `backward_step` method with the `grad_output` argument to calculate the gradients for the inputs
- call the `backward` method of the inputs with the calculated gradient in the current node as an argument
- reset the `n_grads` counter to 0 (actually, this is not necessary as we do backpropagation only once)

Also `backward_step` method should be implemented for basic node too. It should return an empty list as basic node does not have any inputs.

In general, we will sotre inputs to the operation as list of nodes in the `inputs` field. The `backward_step` function should return the corresponding list of gradients for the inputs.

After these steps are done, you can already check that the first test passes successfully

# Task 2

Implement the `__add__` method for the `Node` class and the class `TensorSum` that inherits from `Node`. 
This method enables to use the `+` operator for a pair of objects of `Node` class and means that after we perofrm `z = x + y`, result
of this operation is also a Node that correspond to the sum of two nodes.

The engine for that operation is `TensorSum` class, here you need to:
- initialize the `Node` class with the `None` argument (empty envelope of the node) in the `__init__` method
- in the `__call__` method (this method is used when we perform `z = x + y`):
    - assign the `inputs` fields of the current node to the `[input1, input2]` to store the inputs
    - set `n_dependents` to the sum of `n_dependents` of the inputs (or simply to 2 as we have only 2 inputs)
    - calculate value of `data` field of the current node using the `self.inputs[0].data + self.inputs[1].data` formula
    - set the `grad` field of the current node to `None` as we don't have any gradient yet
    - return the `self` node (so that the result of the operation is the current node)
- in the `backward_step` method:
    - return the list of gradients for the inputs: `[grad_output, grad_output]` as $\partial_x {x + y} = 1$ and $\partial_y {x + y} = 1$


In `Node` class:
- implement the `__add__` method:
    - create a new `TensorSum` object with the sum of the `data` of the current node and the `other` node and return it (`return TensorSum()(self, other)`)

Now your Node class supports the `+` operator. You can check it by runnint test number 2. So you can calculate gradients of expressions like `x + y + x + x + y + y`. That is no much, but that is a good start.

# Task 3 and others

Implement the remaining operations by analogy:
- `__sub__` that will be used when we perform `z = x - y`
- `__mul__` that will be used when we perform `z = x * y`
- `__truediv__` that will be used when we perform `z = x / y`
- `__neg__` that will be used when we perform `z = -x`
- `exp` that will be used when we perform `z = numpy.exp(x)`
- `sum` that will be used when we perform `z = numpy.sum(x)`
- `matmul` that will be used when we perform `z = x @ y`

In all these cases, you should use theoretircal derivatives to implement `backward_step` function for every of these operations.

The hardest one will be `matmul` as it is a matrix multiplication. To implement it, use the following rule of thumb:
- the resulting gradient should have the same shape as data
- the gradients of linear functions are also some linear functions

In [18]:
class Node:
    def __init__(self, data):
        if not isinstance(data, np.ndarray):
            data = np.array(data)
        self.data = data
        self.grad = None
        self.inputs = []
        self.n_dependents = 0
        self.n_grads = 0

    def update_grad(self, grad_output):
        grad_output = copy.deepcopy(grad_output)
        self.n_grads += 1
        if grad_output is None and self.data.size == 1:
            # Use the same dtype as the data - create array correctly
            grad_output = np.ones_like(self.data, dtype=self.data.dtype)
            if self.data.shape == ():
                grad_output = np.array([1], dtype=self.data.dtype)
        elif grad_output is None:
            raise BaseException("Backpropagation is impossible")
        
        # For mathematical operations that naturally produce float64 (division, exp),
        # keep the float64 dtype. Otherwise, ensure grad_output has the same dtype as self.data
        if hasattr(grad_output, 'dtype') and grad_output.dtype == np.float64:
            # Keep float64 gradients as they are - they come from mathematical operations
            pass
        elif hasattr(grad_output, 'dtype') and grad_output.dtype != self.data.dtype:
            grad_output = grad_output.astype(self.data.dtype)
        
        if self.grad is None:
            self.grad = grad_output
        else:
            # When accumulating gradients, convert to the common dtype
            if hasattr(self.grad, 'dtype') and hasattr(grad_output, 'dtype'):
                if self.grad.dtype == np.float64 or grad_output.dtype == np.float64:
                    # If either gradient is float64, keep as float64
                    self.grad = self.grad.astype(np.float64) + grad_output.astype(np.float64)
                else:
                    self.grad = self.grad + grad_output
            else:
                self.grad = self.grad + grad_output

    def backward(self, grad_output=None):
        self.update_grad(grad_output)
        if self.n_dependents == 0 or self.n_grads == self.n_dependents:
            if len(self.inputs) > 0:
                grads = self.backward_step()
                for inp, grad in zip(self.inputs, grads):
                    if grad is not None:
                        inp.backward(grad)
            self.n_grads = 0

    def backward_step(self):
        return []

    def __neg__(self):
        return TensorNeg()(self)

    def __add__(self, other):
        return TensorSum()(self, other)

    def __mul__(self, other):
        return TensorMul()(self, other)

    def __sub__(self, other):
        return TensorSub()(self, other)

    def __truediv__(self, other):
        return TensorDiv()(self, other)

    def exp(self):
        return TensorExp()(self)

    def sum(self, axis=None):
        return TensorSumReduce()(self)

    def __matmul__(self, other):
        return TensorMatMul()(self, other)

class TensorSum(Node):
    def __init__(self):
        super().__init__(None)

    def __call__(self, input1, input2):
        self.inputs = [input1, input2]
        input1.n_dependents += 1
        input2.n_dependents += 1
        self.data = input1.data + input2.data
        self.grad = None
        self.input1_shape = input1.data.shape
        self.input2_shape = input2.data.shape
        return self

    def backward_step(self):
        grad1 = broadcast_reduce(self.grad, self.input1_shape)
        grad2 = broadcast_reduce(self.grad, self.input2_shape)
        return [grad1, grad2]

class TensorSub(Node):
    def __init__(self):
        super().__init__(None)

    def __call__(self, input1, input2):
        self.inputs = [input1, input2]
        input1.n_dependents += 1
        input2.n_dependents += 1
        self.data = input1.data - input2.data
        self.grad = None
        self.input1_shape = input1.data.shape
        self.input2_shape = input2.data.shape
        return self

    def backward_step(self):
        grad1 = broadcast_reduce(self.grad, self.input1_shape, self.inputs[0].data.dtype)
        grad2 = broadcast_reduce(-self.grad, self.input2_shape, self.inputs[1].data.dtype)
        return [grad1, grad2]

class TensorMul(Node):
    def __init__(self):
        super().__init__(None)

    def __call__(self, input1, input2):
        self.inputs = [input1, input2]
        input1.n_dependents += 1
        input2.n_dependents += 1
        self.data = input1.data * input2.data
        self.grad = None
        self.input1_shape = input1.data.shape
        self.input2_shape = input2.data.shape
        return self

    def backward_step(self):
        grad1 = self.grad * self.inputs[1].data
        grad2 = self.grad * self.inputs[0].data
        grad1_reduced = broadcast_reduce(grad1, self.input1_shape, self.inputs[0].data.dtype)
        grad2_reduced = broadcast_reduce(grad2, self.input2_shape, self.inputs[1].data.dtype)
        return [grad1_reduced, grad2_reduced]

class TensorDiv(Node):
    def __init__(self):
        super().__init__(None)

    def __call__(self, input1, input2):
        self.inputs = [input1, input2]
        input1.n_dependents += 1
        input2.n_dependents += 1
        # Division should produce float64 data since division can create fractions
        self.data = (input1.data / input2.data).astype(np.float64)
        self.grad = None
        self.input1_shape = input1.data.shape
        self.input2_shape = input2.data.shape
        return self

    def backward_step(self):
        grad1 = self.grad / self.inputs[1].data
        grad2 = -self.grad * self.data / self.inputs[1].data
        # Since self.data is already float64, gradients will be float64
        grad1_reduced = broadcast_reduce(grad1, self.input1_shape, np.float64)
        grad2_reduced = broadcast_reduce(grad2, self.input2_shape, np.float64)
        return [grad1_reduced, grad2_reduced]

class TensorNeg(Node):
    def __init__(self):
        super().__init__(None)

    def __call__(self, input1):
        self.inputs = [input1]
        input1.n_dependents += 1
        self.data = -input1.data
        self.grad = None
        return self

    def backward_step(self):
        grad = -self.grad
        if hasattr(grad, 'astype'):
            grad = grad.astype(self.inputs[0].data.dtype)
        return [grad]

class TensorExp(Node):
    def __init__(self):
        super().__init__(None)

    def __call__(self, input1):
        self.inputs = [input1]
        input1.n_dependents += 1
        # Exponential should produce float64 data since exp produces transcendental numbers
        self.data = np.exp(input1.data).astype(np.float64)
        self.grad = None
        return self

    def backward_step(self):
        grad = self.grad * self.data
        # Since self.data is already float64, grad will be float64
        return [grad]

class TensorSumReduce(Node):
    def __init__(self):
        super().__init__(None)

    def __call__(self, input1):
        self.inputs = [input1]
        input1.n_dependents += 1
        self.data = np.array(np.sum(input1.data))
        self.grad = None
        return self

    def backward_step(self):
        grad = self.grad * np.ones_like(self.inputs[0].data)
        return [grad.astype(self.inputs[0].data.dtype)]

class TensorMatMul(Node):
    def __init__(self):
        super().__init__(None)

    def __call__(self, input1, input2):
        self.inputs = [input1, input2]
        input1.n_dependents += 1
        input2.n_dependents += 1
        self.data = np.matmul(input1.data, input2.data)
        self.grad = None
        return self

    def backward_step(self):
        grad_input1 = np.matmul(self.grad, self.inputs[1].data.T)
        grad_input2 = np.matmul(self.inputs[0].data.T, self.grad)
        return [grad_input1, grad_input2]

In [25]:
# TEST 1

x = Node(numpy.array(1))
print(x)
x.backward()
print(x)

answer['init'] = []
for inp in inputs1:
    x = Node(inp['x'][0])

    x.backward()

    answer['init'].append(x.grad)

In [43]:
# TEST 2

x = Node(numpy.array([1]))
y = Node(numpy.array([2]))

z = x + y + x + x + x + y

print(z)
z.backward()
print(x, y)

answer['sum'] = []
for inp in inputs1:
    x = Node(inp['x'][0])
    y = Node(inp['x'][1])

    z = x + y + x + x + x + y
    z.backward()

    answer['sum'].append(x.grad)

<__main__.Node object at 0xffff1c656f90> <__main__.Node object at 0xffff1c6568d0>


In [46]:
# TEST 3

x = Node(numpy.array([1]))
y = Node(numpy.array([2]))

z = x - y + x + x - x - y

print(z)
z.backward()
print(x, y)

answer['diff'] = []
for inp in inputs1:
    x = Node(inp['x'][0])
    y = Node(inp['x'][1])

    z = x - y + x + x - y - y
    z.backward()

    answer['diff'].append(x.grad)

<__main__.Node object at 0xffff1c695410> <__main__.Node object at 0xffff1c695610>


In [27]:
# TEST 4

x = Node(numpy.array([1]))
y = Node(numpy.array([2]))

z = (x + y) * (x - y)

print(z)
z.backward()
print(x, y)

answer['mul'] = []
for inp in inputs1:
    x = Node(inp['x'][0])
    y = Node(inp['x'][1])

    z = (x + y) * (x - y) * (x + x + y)
    z.backward()

    answer['mul'].append(x.grad)

<__main__.Node object at 0xffff1cb611d0> <__main__.Node object at 0xffff1cb61f10>


In [7]:
# TEST 5

x = Node(numpy.array([1]))
y = Node(numpy.array([2]))

z = x / y

print(z)
z.backward()
print(x, y)

answer['div'] = []
for inp in inputs1:
    x = Node(inp['x'][0])
    y = Node(inp['x'][1])

    z = x / (Node(0.5) + y)
    z.backward()

    answer['div'].append(x.grad)

<__main__.Node object at 0xffff6c063810> <__main__.Node object at 0xffff6c03d1d0>


In [10]:
# TEST 6

x = Node(numpy.array([1]))
y = Node(numpy.array([2]))

z = -x

print(z)
z.backward()
print(x)

answer['neg'] = []
for inp in inputs1:
    x = Node(inp['x'][0])

    z = -x
    z.backward()

    answer['neg'].append(x.grad)

In [8]:
# TEST 7

x = Node(numpy.array([1]))
y = Node(numpy.array([2]))

z = (x + y).exp()

print(z)
print(x.grad, y.grad)

z.backward()

print(x.grad, y.grad)

answer['exp'] = []
for inp in inputs1:
    x = Node(inp['x'][0])

    z = x.exp()
    z.backward()

    answer['exp'].append(x.grad)

None None
[20] [20]


In [16]:
# Test dtype of division and exp gradients
x_div = Node(numpy.array([1]))
y_div = Node(numpy.array([2]))
z_div = x_div / y_div
z_div.backward()
print(f"Division: x.grad.dtype = {x_div.grad.dtype}, y.grad.dtype = {y_div.grad.dtype}")

x_exp = Node(numpy.array([1]))
z_exp = x_exp.exp()
z_exp.backward()
print(f"Exponential: x.grad.dtype = {x_exp.grad.dtype}")

AttributeError: 'TensorDiv' object has no attribute 'backward'

# Task

Implement Graph function from the previous task. For the constants please use `Node(1)` whenever needed (otherwise you will get an error)

In [12]:
# TEST 8 (Graph)

def sigmoid(z):
    return Node(1.0) / (Node(1.0) + (-z).exp())


def tanh(z):
    return (z.exp() - (-z).exp()) / (z.exp() + (-z).exp())

def graph_value(x, w):
    # Assuming * and + are overloaded to use TensorMul and TensorSum
    term1 = TensorMul()(w[0], x[0])
    term2 = TensorMul()(w[1], x[1])
    term3 = TensorMul()(w[2], TensorMul()(x[0], x[1]))
    term4 = w[3]
    sum1 = TensorSum()(term1, term2)
    sum2 = TensorSum()(sum1, term3)
    y = TensorSum()(sum2, term4)
    return y

answer['graph'] = []
for inp in inputs1:

    x = inp['x']
    w = inp['w']

    x = [Node(float(val)) for val in x]
    w = [Node(float(val)) for val in w]

    y = graph_value(x, w)
    y.backward()

    answer['graph'].append([x[0].grad, x[1].grad, w[0].grad, w[1].grad, w[2].grad, w[3].grad])

print("OUR RESULTS:")
print(x[0].grad, x[1].grad, w[0].grad, w[1].grad, w[2].grad, w[3].grad)

def torch_graph_value(x, w):
    y = w[0] * x[0] + w[1] * x[1] + w[2] * x[0] * x[1] + w[3]
    return y

import torch
x = torch.tensor(inp['x'], requires_grad=True, dtype=float)
w = torch.tensor(inp['w'], requires_grad=True, dtype=float)

y = torch_graph_value(x, w)
y.backward()

print("TORCH RESULTS:")
print(x.grad, w.grad)
    

OUR RESULTS:
20.0 -26.0 -7.0 6.0 -42.0 1
TORCH RESULTS:
tensor([ 20., -26.], dtype=torch.float64) tensor([ -7.,   6., -42.,   1.], dtype=torch.float64)
TORCH RESULTS:
tensor([ 20., -26.], dtype=torch.float64) tensor([ -7.,   6., -42.,   1.], dtype=torch.float64)


# 2-layer NN

Implement 2 layer Neural Network and compute its gradient using `Node` class:

$\mathbf y = \sigma( W_2 \sigma(W_1 \mathbf x + \mathbf b_1) + \mathbf b_2)$

Return sum of all values in $y * y$ as loss function

In [13]:
# TEST 9 (Two-layer net)

def two_layer_net(x, W1, W2, b1, b2):
    h = sigmoid(W1 @ x + b1)
    y = sigmoid(W2 @ h + b2)
    return y.sum()

answer['two_layer_net'] = []
for inp in inputs2:
    x = Node(inp['x'])
    W1 = Node(inp['W1'])
    W2 = Node(inp['W2'])
    b1 = Node(inp['b1'])
    b2 = Node(inp['b2'])

    h_hat = two_layer_net(x, W1, W2, b1, b2)
    h_hat.backward()

    answer['two_layer_net'].append([x.grad, W1.grad, W2.grad, b1.grad, b2.grad])

# Conclusion

You have implemented a backpropagation algorithm. This algorithm is similar to one that is used in Torch. Note that you have implemented all the mechanics of it. Thus it should be now not a magical box: you know exactly how it works.

In [29]:
json_tricks.dump(answer, '.answer.json')

'{"sum": [{"__ndarray__": [4.0], "dtype": "float64", "shape": [1]}, {"__ndarray__": [4.0], "dtype": "float64", "shape": [1]}, {"__ndarray__": [4.0], "dtype": "float64", "shape": [1]}, {"__ndarray__": [4.0], "dtype": "float64", "shape": [1]}, {"__ndarray__": [4.0], "dtype": "float64", "shape": [1]}, {"__ndarray__": [4.0], "dtype": "float64", "shape": [1]}, {"__ndarray__": [4.0], "dtype": "float64", "shape": [1]}, {"__ndarray__": [4.0], "dtype": "float64", "shape": [1]}, {"__ndarray__": [4.0], "dtype": "float64", "shape": [1]}, {"__ndarray__": [4.0], "dtype": "float64", "shape": [1]}, {"__ndarray__": [4.0], "dtype": "float64", "shape": [1]}, {"__ndarray__": [4.0], "dtype": "float64", "shape": [1]}, {"__ndarray__": [4.0], "dtype": "float64", "shape": [1]}, {"__ndarray__": [4.0], "dtype": "float64", "shape": [1]}, {"__ndarray__": [4.0], "dtype": "float64", "shape": [1]}, {"__ndarray__": [4.0], "dtype": "float64", "shape": [1]}, {"__ndarray__": [4.0], "dtype": "float64", "shape": [1]}, {"__

In [28]:
# Quick debug: check for any Python floats in answer
print("Checking for problematic types in answer...")
for key, values in answer.items():
    for i, val in enumerate(values[:2]):  # Check first 2 items
        if isinstance(val, list):
            for j, elem in enumerate(val):
                if isinstance(elem, (float, int)) and not isinstance(elem, np.ndarray):
                    print(f"PROBLEM: {key}[{i}][{j}] = {type(elem)} {elem}")
                elif not hasattr(elem, 'dtype'):
                    print(f"PROBLEM: {key}[{i}][{j}] = {type(elem)} (no dtype)")
        else:
            if isinstance(val, (float, int)) and not isinstance(val, np.ndarray):
                print(f"PROBLEM: {key}[{i}] = {type(val)} {val}")
            elif not hasattr(val, 'dtype'):
                print(f"PROBLEM: {key}[{i}] = {type(val)} (no dtype)")
        if i >= 1:  # Only check first 2
            break

Checking for problematic types in answer...


In [30]:
# Check a few sample gradients to verify dtype correctness
print("Sample gradient dtypes:")
for key in ['sum', 'diff', 'mul']:
    if key in answer and len(answer[key]) > 0:
        sample_grad = answer[key][0]
        print(f"{key}: type={type(sample_grad)}, dtype={sample_grad.dtype if hasattr(sample_grad, 'dtype') else 'N/A'}")
        
# Also check the input data types for comparison
sample_input = inputs1[0]['x'][0]
print(f"Input data: type={type(sample_input)}, dtype={sample_input.dtype if hasattr(sample_input, 'dtype') else 'N/A'}")

Sample gradient dtypes:
sum: type=<class 'numpy.ndarray'>, dtype=float64
mul: type=<class 'numpy.ndarray'>, dtype=float64
Input data: type=<class 'numpy.int64'>, dtype=int64


In [31]:
# Check the data types from the input files
print("Input data from inputs1:")
for i in range(3):
    x_val = inputs1[i]['x'][0]
    print(f"  inputs1[{i}]['x'][0]: {x_val}, type: {type(x_val)}, dtype: {getattr(x_val, 'dtype', 'N/A')}")

# Create a test Node to see what happens to dtype
test_node = Node(inputs1[0]['x'][0])
print(f"Node.data: {test_node.data}, dtype: {test_node.data.dtype}")

# Test backpropagation on this node
test_node.backward()
print(f"Node.grad: {test_node.grad}, dtype: {test_node.grad.dtype}")

Input data from inputs1:
  inputs1[0]['x'][0]: 2, type: <class 'numpy.int64'>, dtype: int64
  inputs1[1]['x'][0]: -1, type: <class 'numpy.int64'>, dtype: int64
  inputs1[2]['x'][0]: -9, type: <class 'numpy.int64'>, dtype: int64
Node.data: 2, dtype: int64
Node.grad: [1.], dtype: float64


In [32]:
# Test a single sum operation step-by-step
x = Node(inputs1[0]['x'][0])
y = Node(inputs1[0]['x'][1])

print(f"x.data dtype: {x.data.dtype}")
print(f"y.data dtype: {y.data.dtype}")

z = x + y + x + x + x + y
print(f"z.data dtype: {z.data.dtype}")

z.backward()

print(f"x.grad dtype: {x.grad.dtype}")
print(f"y.grad dtype: {y.grad.dtype}")
print(f"x.grad value: {x.grad}")
print(f"y.grad value: {y.grad}")

x.data dtype: int64
y.data dtype: int64
z.data dtype: int64
x.grad dtype: float64
y.grad dtype: float64
x.grad value: [4.]
y.grad value: [2.]


In [33]:
# Test single operation with detailed tracing
x = Node(inputs1[0]['x'][0])
print(f"1. x.data dtype: {x.data.dtype}")

# Just backward on a single node
x.backward()
print(f"2. x.grad dtype: {x.grad.dtype}")
print(f"   x.grad value: {x.grad}")

# Now test an addition
x2 = Node(inputs1[0]['x'][0])
y2 = Node(inputs1[0]['x'][1])
z2 = x2 + y2

print(f"3. After addition:")
print(f"   x2.data dtype: {x2.data.dtype}")
print(f"   y2.data dtype: {y2.data.dtype}")
print(f"   z2.data dtype: {z2.data.dtype}")

z2.backward()
print(f"4. After backward:")
print(f"   x2.grad dtype: {x2.grad.dtype}")
print(f"   y2.grad dtype: {y2.grad.dtype}")
print(f"   x2.grad value: {x2.grad}")
print(f"   y2.grad value: {y2.grad}")

1. x.data dtype: int64
2. x.grad dtype: float64
   x.grad value: [1.]
3. After addition:
   x2.data dtype: int64
   y2.data dtype: int64
   z2.data dtype: int64
4. After backward:
   x2.grad dtype: float64
   y2.grad dtype: float64
   x2.grad value: [1.]
   y2.grad value: [1.]


In [34]:
# Test array creation directly
test_data = inputs1[0]['x'][0]
print(f"test_data: {test_data}, dtype: {test_data.dtype}")

# Test creating gradient array
test_grad = np.array([1], dtype=test_data.dtype)
print(f"test_grad: {test_grad}, dtype: {test_grad.dtype}")

# Test what happens in Node creation
test_node = Node(test_data)
print(f"test_node.data: {test_node.data}, dtype: {test_node.data.dtype}")

# Now let's trace what happens in update_grad manually
grad_output = None
n_grads = 1
if grad_output is None and test_node.data.size == 1:
    grad_output = np.array([1], dtype=test_node.data.dtype)
    print(f"Created grad_output: {grad_output}, dtype: {grad_output.dtype}")

if test_node.grad is None:
    test_node.grad = grad_output
    print(f"Assigned to grad: {test_node.grad}, dtype: {test_node.grad.dtype}")
else:
    test_node.grad += grad_output

test_data: 2, dtype: int64
test_grad: [1], dtype: int64
test_node.data: 2, dtype: int64
Created grad_output: [1], dtype: int64
Assigned to grad: [1], dtype: int64


In [35]:
# Test the actual update_grad method step by step
test_node2 = Node(inputs1[0]['x'][0])
print(f"Before update_grad: test_node2.data.dtype = {test_node2.data.dtype}")

# Call update_grad with None (simulating backward on leaf node)
grad_output = None
grad_output_copy = copy.deepcopy(grad_output)
print(f"After deepcopy: grad_output_copy = {grad_output_copy}")

test_node2.n_grads += 1
if grad_output_copy is None and test_node2.data.size == 1:
    grad_output_copy = np.array([1], dtype=test_node2.data.dtype)
    print(f"Created grad_output_copy: {grad_output_copy}, dtype: {grad_output_copy.dtype}")

# Check dtype conversion
if hasattr(grad_output_copy, 'dtype') and grad_output_copy.dtype != test_node2.data.dtype:
    print(f"Converting dtype from {grad_output_copy.dtype} to {test_node2.data.dtype}")
    grad_output_copy = grad_output_copy.astype(test_node2.data.dtype)

if test_node2.grad is None:
    test_node2.grad = grad_output_copy
    print(f"Final grad: {test_node2.grad}, dtype: {test_node2.grad.dtype}")

# Now let's call the actual method to compare
test_node3 = Node(inputs1[0]['x'][0])
test_node3.update_grad(None)
print(f"Actual method result: {test_node3.grad}, dtype: {test_node3.grad.dtype}")

Before update_grad: test_node2.data.dtype = int64
After deepcopy: grad_output_copy = None
Created grad_output_copy: [1], dtype: int64
Final grad: [1], dtype: int64
Actual method result: [1.], dtype: float64


In [36]:
# Test different ways of creating arrays
test_data = inputs1[0]['x'][0]
print(f"test_data: {test_data}, dtype: {test_data.dtype}")

# Method 1: np.array([1], dtype=...)
arr1 = np.array([1], dtype=test_data.dtype)
print(f"np.array([1], dtype=int64): {arr1}, dtype: {arr1.dtype}")

# Method 2: direct integer
arr2 = np.array([1])
print(f"np.array([1]): {arr2}, dtype: {arr2.dtype}")

# Method 3: ones_like
arr3 = np.ones_like(test_data)
print(f"np.ones_like(test_data): {arr3}, dtype: {arr3.dtype}")

# Method 4: using the exact value
arr4 = np.array([1], dtype=np.int64)
print(f"np.array([1], dtype=np.int64): {arr4}, dtype: {arr4.dtype}")

# Method 5: create then convert
arr5 = np.array([1.0]).astype(test_data.dtype)
print(f"np.array([1.0]).astype(int64): {arr5}, dtype: {arr5.dtype}")

test_data: 2, dtype: int64
np.array([1], dtype=int64): [1], dtype: int64
np.array([1]): [1], dtype: int64
np.ones_like(test_data): 1, dtype: int64
np.array([1], dtype=np.int64): [1], dtype: int64
np.array([1.0]).astype(int64): [1], dtype: int64


In [37]:
# Step through exactly what the update_grad method does
class DebugNode(Node):
    def update_grad(self, grad_output):
        print(f"  Input grad_output: {grad_output}, type: {type(grad_output)}")
        grad_output = copy.deepcopy(grad_output)
        print(f"  After deepcopy: {grad_output}, type: {type(grad_output)}")
        self.n_grads += 1
        if grad_output is None and self.data.size == 1:
            print(f"  self.data.dtype: {self.data.dtype}")
            grad_output = np.array([1], dtype=self.data.dtype)
            print(f"  Created grad_output: {grad_output}, dtype: {grad_output.dtype}")
        elif grad_output is None:
            raise BaseException("Backpropagation is impossible")
        
        # Ensure grad_output has the same dtype as self.data
        if hasattr(grad_output, 'dtype') and grad_output.dtype != self.data.dtype:
            print(f"  Converting dtype from {grad_output.dtype} to {self.data.dtype}")
            grad_output = grad_output.astype(self.data.dtype)
        
        if self.grad is None:
            print(f"  Assigning grad_output to self.grad")
            self.grad = grad_output
            print(f"  Final self.grad: {self.grad}, dtype: {self.grad.dtype}")
        else:
            print(f"  Adding to existing grad")
            self.grad += grad_output
            print(f"  Final self.grad: {self.grad}, dtype: {self.grad.dtype}")

# Test with debug node
debug_node = DebugNode(inputs1[0]['x'][0])
print(f"Before: debug_node.data = {debug_node.data}, dtype = {debug_node.data.dtype}")
debug_node.update_grad(None)
print(f"After: debug_node.grad = {debug_node.grad}, dtype = {debug_node.grad.dtype}")

Before: debug_node.data = 2, dtype = int64
  Input grad_output: None, type: <class 'NoneType'>
  After deepcopy: None, type: <class 'NoneType'>
  self.data.dtype: int64
  Created grad_output: [1], dtype: int64
  Assigning grad_output to self.grad
  Final self.grad: [1], dtype: int64
After: debug_node.grad = [1], dtype = int64


In [38]:
# Compare regular Node vs DebugNode
regular_node = Node(inputs1[0]['x'][0])
debug_node2 = DebugNode(inputs1[0]['x'][0])

print("Regular Node:")
regular_node.update_grad(None)
print(f"Regular result: {regular_node.grad}, dtype: {regular_node.grad.dtype}")

print("\nDebug Node:")
debug_node2.update_grad(None)
print(f"Debug result: {debug_node2.grad}, dtype: {debug_node2.grad.dtype}")

# Check if there's a difference in how nodes are created
print(f"\nRegular node data: {regular_node.data}, dtype: {regular_node.data.dtype}")
print(f"Debug node data: {debug_node2.data}, dtype: {debug_node2.data.dtype}")

Regular Node:
Regular result: [1.], dtype: float64

Debug Node:
  Input grad_output: None, type: <class 'NoneType'>
  After deepcopy: None, type: <class 'NoneType'>
  self.data.dtype: int64
  Created grad_output: [1], dtype: int64
  Assigning grad_output to self.grad
  Final self.grad: [1], dtype: int64
Debug result: [1], dtype: int64

Regular node data: 2, dtype: int64
Debug node data: 2, dtype: int64


In [1]:
# Test the updated Node implementation
test_node_new = Node(inputs1[0]['x'][0])
print(f"Before: test_node_new.data = {test_node_new.data}, dtype = {test_node_new.data.dtype}")
test_node_new.update_grad(None)
print(f"After: test_node_new.grad = {test_node_new.grad}, dtype = {test_node_new.grad.dtype}")

# Test with backward
test_node_backward = Node(inputs1[0]['x'][0])
test_node_backward.backward()
print(f"Backward result: {test_node_backward.grad}, dtype = {test_node_backward.grad.dtype}")

NameError: name 'Node' is not defined